# Dual-Phase Microstructure Generator

Procedurally generates intricate SEM-like microstructure images of dual-phase steel with a **target martensite/ferrite phase fraction**.

## Physics basis
- **Ferrite** (BCC, light phase in SEM BSE): equiaxed polygonal grains, Voronoi tessellation
- **Martensite** (BCT, dark phase in SEM BSE): lath-like sub-structure within prior austenite grain boundaries, needle/plate morphology
- Grain boundaries, triple junctions, and lath sub-boundaries are rendered explicitly
- Gaussian noise + mild blur simulate SEM detector noise and depth-of-field blur

## Controls
- `martensite_fraction` — target volume fraction of martensite (0.0–1.0)
- `n_ferrite_grains` — number of ferrite parent grains
- `n_austenite_grains` — number of prior austenite grains (martensite islands)
- `image_size` — output pixel resolution (square)
- `seed` — reproducibility
- `magnification` — cosmetic scale bar label


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.spatial import Voronoi
from scipy.ndimage import gaussian_filter, label as nd_label
from PIL import Image, ImageDraw, ImageFont
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

## Core generation utilities

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Utility: rasterise a Voronoi diagram onto a pixel canvas
# ──────────────────────────────────────────────────────────────────────────────

def voronoi_region_map(seeds: np.ndarray, size: int) -> np.ndarray:
    """Return an (size x size) integer array where each pixel holds
    the index of the nearest seed (i.e., Voronoi cell id)."""
    xs, ys = np.meshgrid(np.arange(size), np.arange(size), indexing='xy')
    # (size*size, 2)
    pixels = np.column_stack([xs.ravel(), ys.ravel()]).astype(np.float32)
    # Batched nearest-seed via broadcast
    diff = pixels[:, None, :] - seeds[None, :, :]   # (N_pix, N_seeds, 2)
    dist2 = (diff ** 2).sum(axis=2)                  # (N_pix, N_seeds)
    region_ids = dist2.argmin(axis=1).reshape(size, size)
    return region_ids


def grain_boundary_mask(region_map: np.ndarray, thickness: int = 1) -> np.ndarray:
    """Boolean mask of pixels that sit on a grain boundary (neighbour has
    different region id)."""
    shifted_x = np.roll(region_map, 1, axis=1)
    shifted_y = np.roll(region_map, 1, axis=0)
    boundary = (region_map != shifted_x) | (region_map != shifted_y)
    if thickness > 1:
        from scipy.ndimage import binary_dilation
        boundary = binary_dilation(boundary, iterations=thickness - 1)
    return boundary


print('Voronoi utilities defined.')

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Utility: martensite lath sub-structure within a parent austenite grain
# ──────────────────────────────────────────────────────────────────────────────

def draw_martensite_laths(
    canvas: np.ndarray,
    mask: np.ndarray,
    rng: np.random.Generator,
    n_packets: int = 4,
    laths_per_packet: int = 8,
    lath_width: float = 1.5,
    boundary_value: float = 0.05,
    lath_value_range: tuple = (0.08, 0.22),
) -> np.ndarray:
    """Render lath martensite sub-structure inside `mask` region on `canvas`.
    Returns modified canvas."""
    ys, xs = np.where(mask)
    if len(ys) == 0:
        return canvas

    size = canvas.shape[0]
    cx, cy = xs.mean(), ys.mean()

    # Each packet has a dominant lath direction (3 Bain variants, 24 KS variants → simplified)
    packet_angles = rng.uniform(0, np.pi, n_packets)

    for p_angle in packet_angles:
        # Slightly perturbed lath directions within this packet
        lath_angles = p_angle + rng.normal(0, 0.05, laths_per_packet)
        lath_values = rng.uniform(*lath_value_range, laths_per_packet)

        for angle, val in zip(lath_angles, lath_values):
            # Direction vector of the lath
            dx, dy = np.cos(angle), np.sin(angle)
            # Normal to lath
            nx, ny = -dy, dx

            # Random offset from grain centroid
            extent = max(xs.max() - xs.min(), ys.max() - ys.min()) * 0.5
            offset = rng.uniform(-extent, extent)

            # Signed distance of each pixel in the mask to this lath plane
            dist = (xs - cx) * nx + (ys - cy) * ny - offset
            in_lath = np.abs(dist) < lath_width

            # Draw lath boundary (dark line) and lath interior (slightly lighter)
            lath_pixels = np.zeros(mask.shape, dtype=bool)
            lath_pixels[ys[in_lath], xs[in_lath]] = True
            lath_pixels &= mask

            # Interior of lath – a specific grey tone
            interior = np.zeros(mask.shape, dtype=bool)
            interior_dist = (xs - cx) * nx + (ys - cy) * ny - offset
            in_interior = np.abs(interior_dist) < (lath_width * 0.5)
            interior[ys[in_interior], xs[in_interior]] = True
            interior &= mask

            canvas[lath_pixels] = val
            # Sub-boundary (thin dark line at lath edge)
            boundary_mask = lath_pixels & ~interior
            canvas[boundary_mask] = boundary_value

    return canvas


print('Martensite lath renderer defined.')

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Utility: ferrite grain interior texture (slight orientation contrast)
# ──────────────────────────────────────────────────────────────────────────────

def ferrite_grain_texture(
    canvas: np.ndarray,
    mask: np.ndarray,
    base_brightness: float,
    rng: np.random.Generator,
    texture_amplitude: float = 0.04,
    scratch_prob: float = 0.3,
) -> np.ndarray:
    """Fill ferrite grain with orientation-contrast-like texture."""
    ys, xs = np.where(mask)
    if len(ys) == 0:
        return canvas

    # Gentle linear gradient (simulates crystallographic channelling contrast)
    angle = rng.uniform(0, np.pi)
    gradient = np.cos(angle) * xs + np.sin(angle) * ys
    g_norm = (gradient - gradient.min()) / (gradient.ptp() + 1e-8)
    brightness = base_brightness + (g_norm - 0.5) * texture_amplitude
    canvas[ys, xs] = brightness

    # Occasional slip/scratch lines
    if rng.random() < scratch_prob:
        scratch_angle = angle + rng.normal(0, 0.1)
        sdx, sdy = np.cos(scratch_angle), np.sin(scratch_angle)
        snx, sny = -sdy, sdx
        cx, cy = xs.mean(), ys.mean()
        offset = rng.uniform(-10, 10)
        dist = np.abs((xs - cx) * snx + (ys - cy) * sny - offset)
        in_scratch = dist < 0.8
        canvas[ys[in_scratch], xs[in_scratch]] = base_brightness - 0.06

    return canvas


print('Ferrite texture renderer defined.')

## Main generator function

In [ ]:
def generate_microstructure(
    martensite_fraction: float = 0.30,
    n_ferrite_grains: int = 40,
    n_austenite_grains: int = 20,
    image_size: int = 512,
    seed: int = 42,
    noise_sigma: float = 3.0,
    detector_noise: float = 0.015,
    boundary_thickness: int = 2,
    magnification: str = '5000x',
    show_scalebar: bool = True,
    return_phase_map: bool = False,
):
    """
    Generate a synthetic SEM BSE-like image of dual-phase martensite/ferrite steel.

    Parameters
    ----------
    martensite_fraction : float
        Target volume fraction of martensite (0–1).
    n_ferrite_grains : int
        Number of ferrite grains in the tessellation.
    n_austenite_grains : int
        Number of prior austenite grains (martensite islands).
    image_size : int
        Output image width/height in pixels.
    seed : int
        RNG seed for reproducibility.
    noise_sigma : float
        Gaussian blur sigma to smooth grain boundaries.
    detector_noise : float
        Amplitude of additive Gaussian noise (SEM detector noise).
    boundary_thickness : int
        Grain boundary line thickness in pixels.
    magnification : str
        Label shown on scale bar.
    show_scalebar : bool
        Whether to overlay a scale bar.
    return_phase_map : bool
        If True, also return a boolean mask where True = martensite.

    Returns
    -------
    img_array : np.ndarray (H, W) float32, range [0, 1]
    phase_map : np.ndarray (H, W) bool  [only if return_phase_map=True]
    actual_fraction : float  measured martensite area fraction
    """
    assert 0.0 <= martensite_fraction <= 1.0, 'martensite_fraction must be in [0, 1]'
    rng = np.random.default_rng(seed)
    S = image_size

    # ── Step 1: Ferrite grain tessellation ───────────────────────────────────
    ferrite_seeds = rng.uniform(0, S, (n_ferrite_grains, 2))
    ferrite_map = voronoi_region_map(ferrite_seeds, S)   # (S, S) int

    # Assign random orientation-dependent brightness per grain
    # Ferrite in BSE: bright phase (~0.55–0.75 normalised)
    ferrite_brightness = rng.uniform(0.55, 0.75, n_ferrite_grains)

    canvas = np.zeros((S, S), dtype=np.float32)

    for gid in range(n_ferrite_grains):
        grain_mask = ferrite_map == gid
        canvas = ferrite_grain_texture(
            canvas, grain_mask, ferrite_brightness[gid], rng
        )

    # ── Step 2: Martensite islands (prior austenite grains) ───────────────────
    # Place austenite seeds – cluster them to mimic real morphology
    # (martensite preferentially forms at ferrite grain boundaries and in
    # carbon-rich regions)
    #
    # Strategy: pick grain boundary pixels preferentially as austenite seed sites
    gb_mask = grain_boundary_mask(ferrite_map, thickness=4)
    gb_ys, gb_xs = np.where(gb_mask)

    if len(gb_ys) < n_austenite_grains:
        # Fall back to random if not enough boundary pixels
        aus_seeds = rng.uniform(0, S, (n_austenite_grains, 2))
    else:
        idx = rng.choice(len(gb_ys), n_austenite_grains, replace=False)
        # Slightly perturb so they spread into grain interiors too
        jitter = rng.normal(0, S * 0.04, (n_austenite_grains, 2))
        aus_seeds = np.column_stack([gb_xs[idx], gb_ys[idx]]).astype(float) + jitter
        aus_seeds = np.clip(aus_seeds, 0, S - 1)

    aus_map = voronoi_region_map(aus_seeds, S)  # (S, S) int

    # ── Step 3: Assign phase to each pixel ────────────────────────────────────
    # Sort austenite grains by their pixel count and assign the largest ones
    # as martensite until we hit the target fraction.
    aus_grain_sizes = np.array([(aus_map == i).sum() for i in range(n_austenite_grains)])
    sorted_ids = np.argsort(-aus_grain_sizes)  # largest first

    total_pixels = S * S
    target_pixels = int(martensite_fraction * total_pixels)

    martensite_phase = np.zeros((S, S), dtype=bool)
    accumulated = 0

    for gid in sorted_ids:
        if accumulated >= target_pixels:
            break
        grain_mask = aus_map == gid
        # Only assign pixels not already claimed; stop when target reached
        remaining_needed = target_pixels - accumulated
        pixel_ys, pixel_xs = np.where(grain_mask)
        if len(pixel_ys) <= remaining_needed:
            martensite_phase[grain_mask] = True
            accumulated += len(pixel_ys)
        else:
            # Partial grain: assign a contiguous sub-region
            # Use a random half-plane cut within the grain
            cx, cy = pixel_xs.mean(), pixel_ys.mean()
            cut_angle = rng.uniform(0, np.pi)
            dx, dy = np.cos(cut_angle), np.sin(cut_angle)
            proj = (pixel_xs - cx) * dx + (pixel_ys - cy) * dy
            order = np.argsort(proj)
            chosen = order[:remaining_needed]
            martensite_phase[pixel_ys[chosen], pixel_xs[chosen]] = True
            accumulated += remaining_needed
            break

    actual_fraction = martensite_phase.sum() / total_pixels

    # ── Step 4: Render martensite lath sub-structure ──────────────────────────
    # Find connected martensite islands and render each one independently
    labeled_mart, n_islands = nd_label(martensite_phase)

    for island_id in range(1, n_islands + 1):
        island_mask = labeled_mart == island_id
        if island_mask.sum() < 20:   # skip tiny specks
            canvas[island_mask] = rng.uniform(0.12, 0.22)
            continue
        # Base martensite BSE intensity (dark phase, ~0.12–0.30)
        base_val = rng.uniform(0.12, 0.28)
        canvas[island_mask] = base_val
        canvas = draw_martensite_laths(canvas, island_mask, rng)

    # ── Step 5: Grain boundaries ──────────────────────────────────────────────
    # Ferrite grain boundaries (dark lines)
    ferrite_gb = grain_boundary_mask(ferrite_map, thickness=boundary_thickness)
    # Austenite/martensite boundary
    mart_boundary = grain_boundary_mask(martensite_phase.astype(int), thickness=boundary_thickness)

    canvas[ferrite_gb & ~martensite_phase] = 0.05  # dark GB on ferrite
    canvas[mart_boundary] = 0.03                   # even darker on martensite edge

    # ── Step 6: Smooth + noise ────────────────────────────────────────────────
    canvas = gaussian_filter(canvas, sigma=noise_sigma * 0.15)
    canvas += rng.normal(0, detector_noise, canvas.shape).astype(np.float32)
    canvas = np.clip(canvas, 0.0, 1.0)

    # ── Step 7: Scale bar overlay ─────────────────────────────────────────────
    if show_scalebar:
        canvas = _add_scalebar(canvas, magnification, S)

    if return_phase_map:
        return canvas, martensite_phase, actual_fraction
    return canvas, actual_fraction


# ── Scale bar helper ──────────────────────────────────────────────────────────

def _add_scalebar(canvas: np.ndarray, magnification: str, S: int) -> np.ndarray:
    """Burn a simple scale bar into the bottom-right corner."""
    bar_len = int(S * 0.15)   # 15% of image width
    bar_h   = max(3, int(S * 0.008))
    margin  = int(S * 0.03)
    x0 = S - margin - bar_len
    y0 = S - margin - bar_h - int(S * 0.035)
    x1 = S - margin
    y1 = y0 + bar_h
    canvas[y0:y1, x0:x1] = 1.0   # white bar
    # Tick marks at ends
    tick_h = bar_h * 3
    canvas[y0 - tick_h:y1, x0:x0 + max(1, bar_h // 2)] = 1.0
    canvas[y0 - tick_h:y1, x1 - max(1, bar_h // 2):x1] = 1.0
    return canvas


print('Generator function defined.')

## Quick test — single image

In [ ]:
img, phase_map, actual_frac = generate_microstructure(
    martensite_fraction=0.30,
    n_ferrite_grains=35,
    n_austenite_grains=18,
    image_size=512,
    seed=7,
    return_phase_map=True,
)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

axes[0].imshow(img, cmap='gray', vmin=0, vmax=1)
axes[0].set_title(f'Synthetic SEM BSE — Martensite {actual_frac:.1%}', fontsize=13)
axes[0].axis('off')

# Phase map overlay
rgb = np.stack([img, img, img], axis=-1)
rgb[phase_map] = [0.85, 0.20, 0.15]   # red tint on martensite
axes[1].imshow(rgb)
red_patch   = mpatches.Patch(color=[0.85, 0.20, 0.15], label=f'Martensite ({actual_frac:.1%})')
grey_patch  = mpatches.Patch(color=[0.65, 0.65, 0.65], label=f'Ferrite ({1-actual_frac:.1%})')
axes[1].legend(handles=[red_patch, grey_patch], loc='lower left', fontsize=10)
axes[1].set_title('Phase map overlay', fontsize=13)
axes[1].axis('off')

plt.tight_layout()
plt.savefig('microstructure_single.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Measured martensite fraction: {actual_frac:.4f}')

## Sweep — varying phase fraction
Generate a grid of images at different martensite volume fractions.

In [ ]:
target_fractions = [0.05, 0.15, 0.30, 0.50, 0.70, 0.90]
n_cols = len(target_fractions)

fig, axes = plt.subplots(2, n_cols, figsize=(3.5 * n_cols, 7.5))
fig.suptitle('Dual-Phase Steel — Martensite/Ferrite Phase Fraction Sweep', fontsize=14, y=1.01)

for col, frac in enumerate(target_fractions):
    img, pm, af = generate_microstructure(
        martensite_fraction=frac,
        n_ferrite_grains=30,
        n_austenite_grains=20,
        image_size=256,
        seed=col * 13 + 5,
        return_phase_map=True,
        show_scalebar=False,
    )

    # Row 0: greyscale SEM-like
    axes[0, col].imshow(img, cmap='gray', vmin=0, vmax=1)
    axes[0, col].set_title(f'M={frac:.0%}\n(actual {af:.1%})', fontsize=9)
    axes[0, col].axis('off')

    # Row 1: phase map
    rgb = np.stack([img, img, img], axis=-1)
    rgb[pm] = [0.85, 0.20, 0.15]
    axes[1, col].imshow(rgb)
    axes[1, col].axis('off')

axes[0, 0].set_ylabel('BSE image', fontsize=9)
axes[1, 0].set_ylabel('Phase map', fontsize=9)

plt.tight_layout()
plt.savefig('microstructure_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sweep complete.')

## Batch generation — produce a labelled dataset
Generate N images per target fraction and save them as PNG + a CSV manifest.

In [ ]:
import os, csv
from pathlib import Path


def batch_generate(
    output_dir: str = 'generated_microstructures',
    fractions: list = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70],
    n_per_fraction: int = 5,
    image_size: int = 512,
    base_seed: int = 0,
):
    """
    Generate `n_per_fraction` images at each target martensite fraction.
    Saves PNGs and a manifest CSV to `output_dir`.
    """
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)

    manifest_path = out / 'manifest.csv'
    rows = []
    total = len(fractions) * n_per_fraction
    done = 0

    for frac in fractions:
        for i in range(n_per_fraction):
            seed = base_seed + done
            img, phase_map, af = generate_microstructure(
                martensite_fraction=frac,
                n_ferrite_grains=rng_n(seed, 25, 50),
                n_austenite_grains=rng_n(seed + 1, 12, 30),
                image_size=image_size,
                seed=seed,
                noise_sigma=np.random.default_rng(seed).uniform(1.5, 4.0),
                return_phase_map=True,
                show_scalebar=True,
            )

            fname = f'micro_{frac:.2f}_{i:03d}.png'
            # Convert float32 → uint8 greyscale
            pil_img = Image.fromarray((img * 255).astype(np.uint8), mode='L')
            pil_img.save(out / fname)

            rows.append({
                'filename': fname,
                'target_martensite_fraction': frac,
                'actual_martensite_fraction': round(af, 5),
                'seed': seed,
                'image_size': image_size,
            })
            done += 1
            if done % 5 == 0 or done == total:
                print(f'  {done}/{total} images written...', end='\r')

    with open(manifest_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)

    print(f'\nDone. {total} images + manifest saved to: {out.resolve()}')
    return manifest_path


def rng_n(seed, lo, hi):
    return int(np.random.default_rng(seed).integers(lo, hi))


print('Batch generator defined.\n'
      'Call batch_generate() below to produce the dataset.')

In [ ]:
# ── Run batch generation (adjust parameters as needed) ────────────────────────
manifest = batch_generate(
    output_dir='generated_microstructures',
    fractions=[0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90],
    n_per_fraction=5,
    image_size=512,
    base_seed=1000,
)

## Inspect generated dataset

In [ ]:
import pandas as pd

df = pd.read_csv(manifest)
print(df.to_string(index=False))

# Accuracy of fraction targeting
df['error'] = (df['actual_martensite_fraction'] - df['target_martensite_fraction']).abs()
print(f'\nMean |target − actual| fraction: {df["error"].mean():.4f}')
print(f'Max  |target − actual| fraction: {df["error"].max():.4f}')

# Quick histogram
fig, ax = plt.subplots(figsize=(7, 3))
ax.scatter(df['target_martensite_fraction'], df['actual_martensite_fraction'],
           alpha=0.7, edgecolors='k', linewidths=0.5)
ax.plot([0, 1], [0, 1], 'r--', lw=1, label='Perfect targeting')
ax.set_xlabel('Target martensite fraction')
ax.set_ylabel('Actual martensite fraction')
ax.set_title('Phase fraction targeting accuracy')
ax.legend()
plt.tight_layout()
plt.show()

## Interactive single-image configurator
Run this cell to explore microstructures interactively via parameter sliders (requires `ipywidgets`).

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    def _interactive_plot(
        martensite_fraction, n_ferrite_grains, n_austenite_grains, seed, noise_sigma
    ):
        img, pm, af = generate_microstructure(
            martensite_fraction=martensite_fraction,
            n_ferrite_grains=n_ferrite_grains,
            n_austenite_grains=n_austenite_grains,
            image_size=384,
            seed=seed,
            noise_sigma=noise_sigma,
            return_phase_map=True,
            show_scalebar=True,
        )
        fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
        axes[0].imshow(img, cmap='gray', vmin=0, vmax=1)
        axes[0].set_title(f'BSE image — M={af:.1%}', fontsize=12)
        axes[0].axis('off')
        rgb = np.stack([img, img, img], axis=-1)
        rgb[pm] = [0.85, 0.20, 0.15]
        axes[1].imshow(rgb)
        axes[1].set_title('Phase map  (red=martensite)', fontsize=12)
        axes[1].axis('off')
        plt.tight_layout()
        plt.show()

    widgets.interact(
        _interactive_plot,
        martensite_fraction=widgets.FloatSlider(value=0.30, min=0.02, max=0.95, step=0.01,
                                                description='M fraction'),
        n_ferrite_grains=widgets.IntSlider(value=35, min=10, max=80, step=5,
                                           description='Ferrite grains'),
        n_austenite_grains=widgets.IntSlider(value=18, min=5, max=50, step=1,
                                             description='Aus. grains'),
        seed=widgets.IntSlider(value=42, min=0, max=500, step=1, description='Seed'),
        noise_sigma=widgets.FloatSlider(value=2.0, min=0.5, max=8.0, step=0.5,
                                        description='Blur σ'),
    )
except ImportError:
    print('ipywidgets not installed — run: pip install ipywidgets')
    print('Using static call instead:\n')
    img, pm, af = generate_microstructure(
        martensite_fraction=0.45,
        n_ferrite_grains=40,
        n_austenite_grains=22,
        image_size=384,
        seed=99,
        return_phase_map=True,
    )
    fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
    axes[0].imshow(img, cmap='gray', vmin=0, vmax=1)
    axes[0].set_title(f'BSE image — M={af:.1%}', fontsize=12)
    axes[0].axis('off')
    rgb = np.stack([img, img, img], axis=-1)
    rgb[pm] = [0.85, 0.20, 0.15]
    axes[1].imshow(rgb)
    axes[1].set_title('Phase map  (red=martensite)', fontsize=12)
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()
    print(f'Martensite fraction: {af:.4f}')